# 在 ASGI 应用程序中集成 FastMCP
将 FastMCP 服务器集成到现有的 Starlette、FastAPI 或其他 ASGI 应用程序中

虽然 FastMCP 提供了独立的服务器功能，但您也可以将 FastMCP 服务器集成到现有的 Web 应用程序中。此方法适用于：

- 向现有网站或 API 添加 MCP 功能
- 在特定的 URL 路径下挂载 MCP 服务器
- 在单个应用程序中组合多个服务
- 利用现有的身份验证和中间件

请注意，所有 FastMCP 服务器都提供run()启动服务器的方法。本指南重点介绍如何与更广泛的 ASGI 框架集成。

## ASGI服务器
FastMCP 服务器可以创建为Starlette ASGI 应用程序，以便直接托管或集成到现有应用程序中。

第一步是使用以下http_app()命令从 FastMCP 服务器获取 Starlette 应用程序实例：


此`http_app()`方法是 FastMCP 2.3.2 中的新方法。在旧版本中，`sse_app()`用于SSE 传输或`streamable_http_app()` 传输Streamable HTTP。

In [ ]:
from fastmcp import FastMCP

mcp = FastMCP("MyServer")

@mcp.tool()
def hello(name: str) -> str:
    return f"Hello, {name}!"

# Get a Starlette app instance for Streamable HTTP transport (recommended)
http_app = mcp.http_app()

# For legacy SSE transport (deprecated)
sse_app = mcp.http_app(transport="sse")

返回的应用程序会将 FastMCP 实例存储在 `app.state.fastmcp_server`，因此您可以通过 `request.app.state.fastmcp_server` 从自定义中间件或路由访问该实例。


MCP服务器的端点安装在root Path`/MCP`中，用于流式`http`传输，而 `/SSE`进行`SSE`传输，尽管您可以通过将路径参数传递到`http_app（）`方法：

In [ ]:
# For Streamable HTTP transport
http_app = mcp.http_app(path="/custom-mcp-path")

# For SSE transport (deprecated)
sse_app = mcp.http_app(path="/custom-sse-path", transport="sse")

## 运行服务器
要运行 FastMCP 服务器，您可以使用uvicornASGI 服务器：

In [ ]:
from fastmcp import FastMCP
import uvicorn

mcp = FastMCP("MyServer")

http_app = mcp.http_app()

if __name__ == "__main__":
    uvicorn.run(http_app, host="0.0.0.0", port=8000)

或者，从命令行：

```shell
uvicorn path.to.your.app:http_app --host 0.0.0.0 --port 8000
```

## 自定义中间件

您可以通过将中间件实例列表传递给应用程序创建方法，将自定义 `Starlette` 中间件添加到 FastMCP ASGI 应用程序中：

In [ ]:
from fastmcp import FastMCP
from starlette.middleware import Middleware
from starlette.middleware.cors import CORSMiddleware

# Create your FastMCP server
mcp = FastMCP("MyServer")

# Define custom middleware
custom_middleware = [
    Middleware(CORSMiddleware, allow_origins=["*"]),
]

# Create ASGI app with custom middleware
http_app = mcp.http_app(middleware=custom_middleware)

## Starlette 集成
您可以在另一个 Starlette 应用程序中安装 FastMCP 服务器：

In [ ]:
from fastmcp import FastMCP
from starlette.applications import Starlette
from starlette.routing import Mount

# Create your FastMCP server as well as any tools, resources, etc.
mcp = FastMCP("MyServer")

# Create the ASGI app
mcp_app = mcp.http_app(path='/mcp')

# Create a Starlette app and mount the MCP server
app = Starlette(
    routes=[
        Mount("/mcp-server", app=mcp_app),
        # Add other routes as needed
    ],
    lifespan=mcp_app.lifespan,
)


MCP 端点`/mcp-server/mcp`将在生成的 Starlette 应用程序中可用。

### 嵌套安装
您可以通过嵌套挂载来创建复杂的路由结构：

In [ ]:
from fastmcp import FastMCP
from starlette.applications import Starlette
from starlette.routing import Mount

# Create your FastMCP server as well as any tools, resources, etc.
mcp = FastMCP("MyServer")

# Create the ASGI app
mcp_app = mcp.http_app(path='/mcp')

# Create nested application structure
inner_app = Starlette(routes=[Mount("/inner", app=mcp_app)])
app = Starlette(
    routes=[Mount("/outer", app=inner_app)],
    lifespan=mcp_app.lifespan,
)

在此设置中，MCP 服务器可通过生成的 Starlette 应用程序的 /outer/inner/mcp 路径访问。

## FastAPI 集成
FastAPI 基于 Starlette 构建，因此您可以以类似的方式安装 FastMCP 服务器：

In [ ]:
from fastmcp import FastMCP
from fastapi import FastAPI
from starlette.routing import Mount

# Create your FastMCP server as well as any tools, resources, etc.
mcp = FastMCP("MyServer")

# Create the ASGI app
mcp_app = mcp.http_app(path='/mcp')

# Create a FastAPI app and mount the MCP server
app = FastAPI(lifespan=mcp_app.lifespan)
app.mount("/mcp-server", mcp_app)

## 自定义路线

除了将 FastMCP 服务器添加到现有的 ASGI 应用之外，您还可以向 FastMCP 服务器添加自定义 Web 路由，这些路由将与 MCP 端点一起公开。为此，请使用`@custom_route`装饰器。请注意，这比使用完整的 ASGI 框架灵活性较低，但对于向独立服务器添加简单的端点（例如健康检查）非常有用。



In [ ]:
from fastmcp import FastMCP
from starlette.requests import Request
from starlette.responses import PlainTextResponse

mcp = FastMCP("MyServer")

@mcp.custom_route("/health", methods=["GET"])
async def health_check(request: Request) -> PlainTextResponse:
    return PlainTextResponse("OK")